# Tech Challenge Fase 2  
## Notebook 02 — Silver Orquestrador

### Responsabilidade do notebook

Este notebook prepara a execução da camada **Silver**.

Sua função é centralizar os metadados dos datasets, validar se todas as partições Bronze necessárias foram geradas e registrar os caminhos oficiais de entrada e saída utilizados pelos notebooks Silver.

A camada Silver é responsável por:

- auditoria dos dados;
- limpeza;
- padronização;
- tipagem;
- tratamento de valores ausentes;
- harmonização de schemas;
- persistência dos dados tratados por ano.

---

### Dependências

```text
00_setup_ambiente
01_bronze_planejador
01_1 a 01_5 — notebooks Bronze
```

### Saída principal

```text
config/silver_metadata
logs/pipeline_execution/silver/
```

## 1. Contexto na Arquitetura Medalhão

```text
Raw
 ↓
Bronze
 ↓
Silver  ← você está aqui
 ↓
Gold
```

A Bronze preserva os dados de origem.  
A Silver aplica regras técnicas de qualidade e padronização, preparando os dados para integração analítica na Gold.

## 2. Importação das bibliotecas

Nesta etapa são importados recursos para:

- leitura do `config.json`;
- criação da tabela de metadados Silver;
- validação das partições Bronze;
- persistência de logs operacionais com schema explícito.

In [0]:
import json

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType
)

## 3. Leitura das configurações oficiais do projeto

Os caminhos são obtidos exclusivamente do arquivo `config.json`, criado no notebook de setup.

Essa abordagem evita caminhos fixos espalhados pelos notebooks e garante consistência entre Bronze, Silver e Gold.

In [0]:
CONFIG_FILE_PATH = "/Volumes/workspace/default/vol_trio_drive/projetos/fiap/tech_challenge_fase2/config/config.json"

config = json.loads(dbutils.fs.head(CONFIG_FILE_PATH))

BASE_PATH = config["environment"]["base_path"]
BRONZE_PATH = config["paths"]["bronze_path"]
SILVER_PATH = config["paths"]["silver_path"]
LOG_PATH = config["paths"]["log_path"]
CONFIG_PATH = config["paths"]["config_path"]
EXECUTION_DATE = config["project"]["execution_date"]

print("BASE_PATH:", BASE_PATH)
print("BRONZE_PATH:", BRONZE_PATH)
print("SILVER_PATH:", SILVER_PATH)
print("LOG_PATH:", LOG_PATH)
print("CONFIG_PATH:", CONFIG_PATH)
print("EXECUTION_DATE:", EXECUTION_DATE)

## 4. Construção da tabela de metadados Silver

Cada registro representa uma partição anual de um dataset.

Os metadados informam:

- dataset;
- ano;
- caminho da partição Bronze;
- caminho da partição Silver;
- arquivo Silver esperado;
- formato de entrada;
- formato de saída;
- notebook responsável.

In [0]:
silver_metadata = []

def add_metadata(
    dataset,
    ano,
    silver_file_name,
    notebook
):
    silver_metadata.append({
        "dataset": dataset,
        "ano": int(ano),
        "bronze_path": f"{BRONZE_PATH}/{dataset}/ano={ano}",
        "silver_path": f"{SILVER_PATH}/{dataset}/ano={ano}",
        "silver_file_name": silver_file_name,
        "input_format": "parquet",
        "output_format": "csv",
        "notebook": notebook
    })

for ano in [2023, 2024, 2025]:
    add_metadata(
        "alunos",
        ano,
        f"TS_ALUNO_{ano}.csv",
        "01_silver_alunos"
    )

    add_metadata(
        "municipios",
        ano,
        f"TS_MUNICIPIO_{ano}.csv",
        "02_silver_municipios"
    )

    add_metadata(
        "estados",
        ano,
        f"TS_ESTADO_{ano}_SILVER.csv",
        "03_silver_estados"
    )

    add_metadata(
        "metas_municipios",
        ano,
        f"TS_METAS_MUNICIPIOS_{ano}_SILVER.csv",
        "04_silver_metas_municipios"
    )

    add_metadata(
        "metas_ufs",
        ano,
        f"TS_METAS_UFS_{ano}_SILVER.csv",
        "05_silver_metas_ufs"
    )

## 5. Persistência da tabela `silver_metadata`

A tabela será armazenada em Parquet na área de configuração do projeto.

Os notebooks filhos consultarão essa tabela para resolver seus caminhos de entrada e saída.

In [0]:
schema_metadata = StructType([
    StructField("dataset", StringType(), True),
    StructField("ano", IntegerType(), True),
    StructField("bronze_path", StringType(), True),
    StructField("silver_path", StringType(), True),
    StructField("silver_file_name", StringType(), True),
    StructField("input_format", StringType(), True),
    StructField("output_format", StringType(), True),
    StructField("notebook", StringType(), True)
])

df_silver_metadata = spark.createDataFrame(
    silver_metadata,
    schema=schema_metadata
)

metadata_path = f"{CONFIG_PATH}/silver_metadata"

(
    df_silver_metadata
    .coalesce(1)
    .write
    .mode("overwrite")
    .format("parquet")
    .option("compression", "snappy")
    .save(metadata_path)
)

display(df_silver_metadata.orderBy("dataset", "ano"))

print("silver_metadata salva em:")
print(metadata_path)

## 6. Validação das partições Bronze

Antes da execução da Silver, verificamos se todas as partições Bronze esperadas existem.

Essa validação evita iniciar o tratamento sobre uma camada Bronze incompleta.

In [0]:
validation_results = []

for item in silver_metadata:
    bronze_partition = item["bronze_path"]

    try:
        files = dbutils.fs.ls(bronze_partition)

        parquet_files = [
            file.path
            for file in files
            if file.name.endswith(".parquet")
        ]

        if parquet_files:
            status = "OK"
            error_message = ""
        else:
            status = "PENDENTE"
            error_message = "Partição encontrada, mas sem arquivo Parquet."
    except Exception as e:
        status = "PENDENTE"
        error_message = str(e)

    validation_results.append({
        "dataset": str(item["dataset"]),
        "ano": int(item["ano"]),
        "bronze_path": str(bronze_partition),
        "status": str(status),
        "error_message": str(error_message),
        "execution_date": str(EXECUTION_DATE)
    })

schema_validation = StructType([
    StructField("dataset", StringType(), True),
    StructField("ano", IntegerType(), True),
    StructField("bronze_path", StringType(), True),
    StructField("status", StringType(), True),
    StructField("error_message", StringType(), True),
    StructField("execution_date", StringType(), True)
])

df_validation = spark.createDataFrame(
    validation_results,
    schema=schema_validation
)

display(df_validation.orderBy("dataset", "ano"))

## 7. Resumo da prontidão da camada Bronze

O resumo apresenta a quantidade de partições disponíveis e pendentes por dataset.

In [0]:
df_validation_summary = (
    df_validation
    .groupBy("dataset")
    .agg(
        F.count("*").alias("particoes_esperadas"),
        F.sum(
            F.when(F.col("status") == "OK", 1).otherwise(0)
        ).alias("particoes_disponiveis"),
        F.sum(
            F.when(F.col("status") == "PENDENTE", 1).otherwise(0)
        ).alias("particoes_pendentes")
    )
    .withColumn(
        "status_dataset",
        F.when(
            F.col("particoes_pendentes") == 0,
            "PRONTO"
        ).otherwise("PENDENTE")
    )
)

display(df_validation_summary.orderBy("dataset"))

## 8. Persistência do log de validação

O resultado será armazenado em logs para auditoria e monitoramento operacional.

In [0]:
validation_log_path = (
    f"{LOG_PATH}/pipeline_execution/silver/"
    f"validacao_bronze_execution_date={EXECUTION_DATE}"
)

(
    df_validation
    .coalesce(1)
    .write
    .mode("overwrite")
    .format("parquet")
    .option("compression", "snappy")
    .save(validation_log_path)
)

print("Log de validação salvo em:")
print(validation_log_path)

## 9. Checklist final

O notebook interrompe a execução se alguma partição Bronze estiver ausente.

In [0]:
pending_partitions = (
    df_validation
    .filter(F.col("status") == "PENDENTE")
    .count()
)

if pending_partitions > 0:
    display(
        df_validation
        .filter(F.col("status") == "PENDENTE")
        .orderBy("dataset", "ano")
    )

    raise Exception(
        f"Existem {pending_partitions} partições Bronze pendentes. "
        "Execute ou corrija os notebooks Bronze antes de seguir."
    )

print("Silver Orquestrador concluído com sucesso.")
print("Todas as partições Bronze estão prontas para tratamento.")

## Resultado esperado

Ao final deste notebook:

```text
config/silver_metadata
logs/pipeline_execution/silver/validacao_bronze_execution_date=YYYY-MM-DD
```

estarão disponíveis.

### Próximos notebooks

```text
01_silver_alunos
02_silver_municipios
03_silver_estados
04_silver_metas_municipios
05_silver_metas_ufs
```